# 01 · The sample: JWST galaxy papers that use JWST data

**Output:** `general/outputs/sample.csv` (the sample), `general/outputs/rejected.csv` (candidates that fail
the rule, with the reason), and two browsable pages in `general/outputs/`.

**The sample rule** (`common/jwst_use.py`). A paper is in the sample when it passes four steps, in this order;
the first step it fails is the reason it is left out.

1. **Candidate.** It matches the ADS candidate query (`CANDIDATE_Q`), published 2023 or later: abstract mentions
   "galaxy" or "galaxies", full text mentions JWST, arXiv class astro-ph.GA or astro-ph.CO (or no astro-ph
   class), and JWST in the title or abstract or a JWST instrument, survey or data phrase in the full text.
2. **Uses JWST data.** It has one of two kinds of evidence:
   - *curated*: it is in ADS's curated JWST bibliography (`bibgroup:JWST`, kept by STScI);
   - *explicit text*: one passage of its text scores **at least 7 points**: 7 for STScI's JWST data
     acknowledgement or a "Facilities: JWST" tag; otherwise +2 data wording ("JWST data / observations /
     spectra ..."), +1 instrument name, +2 survey, programme, MAST or GO identifier, +2 "we use" or past-tense
     wording, +2 data claimed in the title or abstract, −3 prediction wording (will, forecast, mock,
     simulation ...). Passages: three ADS highlight passes over the full text, plus each abstract sentence
     naming JWST, an instrument or a survey.
3. **Journal article or preprint** (ADS document type `article` or `eprint`).
4. **Galaxy paper.** Primary arXiv class not astro-ph.SR or astro-ph.EP; and if the galaxy screen flags it
   (non-galaxy words in the title or abstract), a hand verdict "galaxy" in `data/<freeze>/galaxy_verdicts.csv`.
   Flagged papers not yet judged stay in, marked "awaiting review" (`general/tools/galaxy_screen.py`).

Records of the same paper (arXiv preprint and published version) are merged first, keeping the published
one. No paper is added or removed by name outside these steps.


In [ ]:
# Setup: the repository root on the import path, shared settings and helpers.
import os, sys, json, html, re
sys.path.insert(0, os.path.abspath(".."))
from common.config import require_texts
require_texts()   # this notebook reads ADS abstracts or full-text passages
import numpy as np
import pandas as pd
from IPython.display import display

from common.config import (FREEZE, YEAR_RANGE, CALC_FROM_YEAR, SORT, GEN_OUT, data)
from common.ads import (ads_get, paginated_search, load_raw, save_raw, dedupe, fetch_field, is_paper)
from common.jwst_use import (CANDIDATE_Q, CANDIDATE_FIELDS, HL_Q, ADS_ABS, DataUseScorer, galaxy_status,
                             load_galaxy_verdicts, screen_terms)
from common.authors import n_authors, first_author

RAW_JSON = data("raw_candidate_search.json")   # the candidate search, with first-pass highlights
CANDIDATE_FQ = [f"year:{YEAR_RANGE}", "collection:astronomy"]
MAX_ROWS = 12000          # cap on candidates fetched; a warning is printed if the query finds more
ROWS_PER_PAGE = 50        # highlight responses are heavy: small pages
print(f"Freeze {FREEZE}; candidate query:\n{CANDIDATE_Q}")

## 1 · Candidates

One ADS search with first-pass highlights (`HL_Q`: instrument, survey and data phrases, or JWST), cached in
`data/<freeze>/raw_candidate_search.json`. On a rerun with the cache in place, no query is made.

In [ ]:
SIG = json.dumps([CANDIDATE_Q, CANDIDATE_FQ, CANDIDATE_FIELDS, HL_Q, MAX_ROWS, SORT], sort_keys=True)
raw = load_raw(RAW_JSON, SIG)
if raw is None:
    n_found = ads_get("/search/query", {"q": CANDIDATE_Q, "fq": CANDIDATE_FQ, "fl": "bibcode", "rows": 1})["response"]["numFound"]
    print(f"{n_found} candidates" + (f"  WARNING: only the first {MAX_ROWS} are fetched; raise MAX_ROWS" if n_found > MAX_ROWS else ""))
    docs, hl = [], {}
    for page, highlighting in paginated_search(q=CANDIDATE_Q, fq=CANDIDATE_FQ, fl=CANDIDATE_FIELDS, rows=ROWS_PER_PAGE,
                                               max_rows=MAX_ROWS, hl=True, hl_fl="body,abstract,title,ack",
                                               hl_q=HL_Q, hl_snippets=5):
        for d in page:
            if str(d.get("id")) in highlighting:
                hl[d["bibcode"]] = highlighting[str(d.get("id"))]
        docs.extend(page)
        print(f"  fetched {len(docs)} candidates", end="\r")
    save_raw(RAW_JSON, SIG, docs, hl)
else:
    docs, hl = raw["docs"], raw["highlighting"]

docs, hl, n_dup = dedupe(docs, hl)
print(f"{len(docs)} candidates after merging {n_dup} duplicate preprint/published records")

## 2 · Scoring and selection

Per-paper lookups, all cached per bibcode in `data/<freeze>/`: document type, full abstract, arXiv classes,
the second (data-use wording) and third (whole body) highlight passes. Then each candidate is scored and
selected by the rule above.

In [ ]:
bibs = [d["bibcode"] for d in docs]
doctypes = fetch_field(bibs, "doctype", data("doctypes.json"))
abstracts = fetch_field(bibs, "abstract", data("abstracts.json"))
arxiv = fetch_field(bibs, "arxiv_class", data("arxiv_class.json"))
scorer = DataUseScorer()
scorer.prepare(bibs, hl, abstracts)
VERDICTS = load_galaxy_verdicts()

rows = []
for d in docs:
    v = scorer.score(d, hl)
    yr = d.get("year", "")
    row = {"bibcode": d["bibcode"], "ads_url": ADS_ABS.format(d["bibcode"]),
           "title": html.unescape((d.get("title") or ["Untitled"])[0]),
           "year": int(yr) if str(yr).isdigit() else None, "citation_count": d.get("citation_count", 0) or 0,
           "n_authors": n_authors(d), "first_author": first_author(d), "authors": " | ".join(d.get("author") or []),
           "doctype": doctypes.get(d["bibcode"]), "identifiers": " | ".join(d.get("identifier") or []), **v}
    # the rule, step by step; the first failed step is the reason a paper is left out
    text = f"{row['title']} {abstracts.get(d['bibcode']) or ''}"
    row["galaxy_screen"] = "; ".join(screen_terms(text))
    if not v["uses_jwst_data"]:
        row["status"] = f"no JWST data-use evidence ({v['evidence']})"
    elif not is_paper(row["doctype"]):
        row["status"] = f"not a journal article or preprint (ADS doctype: {row['doctype']})"
    else:
        ok, why = galaxy_status(d["bibcode"], arxiv.get(d["bibcode"]), text, VERDICTS)
        row["galaxy_check"] = why or ("judged galaxy" if row["galaxy_screen"] else "")
        row["status"] = "in sample" if ok else f"not a galaxy paper: {why}"
    rows.append(row)
allc = pd.DataFrame(rows)
sample = allc[allc["status"] == "in sample"].drop(columns="status").reset_index(drop=True)
rejected = allc[allc["status"] != "in sample"].reset_index(drop=True)
print(f"Sample: {len(sample):,} papers; rejected: {len(rejected):,}")
display(rejected["status"].str.replace(r":.*", "", regex=True).value_counts().rename("candidates").to_frame())
display(sample.groupby("year").size().rename("papers").to_frame().T)
display(sample["evidence"].value_counts().rename("papers").to_frame().T)
print(f"Galaxy screen: {(sample['galaxy_check'] == 'awaiting review').sum()} sample papers flagged and awaiting a "
      f"verdict (general/tools/galaxy_screen.py page)")

## 3 · Export

`sample.csv` and `rejected.csv` (one row per paper: metadata, data-use evidence and score, the passages that
decided it, the galaxy-screen terms), and two sortable HTML pages to review them, with JWST terms highlighted in blue and prediction
wording in red.

In [ ]:
sample.to_csv(os.path.join(GEN_OUT, "sample.csv"), index=False)
rejected.to_csv(os.path.join(GEN_OUT, "rejected.csv"), index=False)

JWST_TERMS = r"\b(JWST|NIRCam|NIRSpec|NIRISS|MIRI)\b"
NEG_TERMS = (r"\b(will|would|could|future|upcoming|forthcoming|planned|proposed|anticipated|prospects?|"
             r"forecasts?|predict\w*|prepar\w*|mock|simulat\w*|synthetic|expected to)\b")
SORT_JS = (
    "document.querySelectorAll('th').forEach(function(th,i){th.style.cursor='pointer';"
    "th.addEventListener('click',function(){var tb=th.closest('table').tBodies[0],rows=Array.from(tb.rows),"
    "asc=th.dataset.asc!=='1';th.dataset.asc=asc?'1':'0';function v(r){var t=r.cells[i].textContent.trim(),"
    "n=parseFloat(t);return isNaN(n)?t.toLowerCase():n;}rows.sort(function(a,b){var x=v(a),y=v(b);"
    "if(typeof x==='number'&&typeof y==='number')return asc?x-y:y-x;if(typeof x==='number')return -1;"
    "if(typeof y==='number')return 1;return asc?x.localeCompare(y):y.localeCompare(x);});"
    "rows.forEach(function(r){tb.appendChild(r);});});});")
CSS = ("body{font-family:sans-serif;margin:1.5em}table{border-collapse:collapse;font-size:13px}"
       "th{background:#2b2b2b;color:#fff;text-align:left;position:sticky;top:0}"
       "th,td{padding:6px 10px;border-bottom:1px solid #ddd;vertical-align:top}"
       "mark.alt{background:#bfdbfe}mark.neg{background:#fecaca}td:last-child{max-width:60em}")

def _hl(text):
    t = re.sub(JWST_TERMS, r"<mark class='alt'><b>\1</b></mark>", html.escape(text))
    return re.sub(NEG_TERMS, r"<mark class='neg'>\1</mark>", t, flags=re.I)

def save_html(df, name, title):
    d = df.drop(columns=["authors", "identifiers"]).copy()
    d["title"] = d["title"].map(html.escape)
    d["ads_url"] = d["ads_url"].map(lambda u: f'<a href="{u}" target="_blank">ADS</a>')
    d["snippets"] = d["snippets"].fillna("").map(lambda t: "<br><br>".join(_hl(x) for x in t.split(" ||| ")) if t else "")
    with open(os.path.join(GEN_OUT, name), "w", encoding="utf-8") as f:
        f.write(f"<!doctype html><meta charset='utf-8'><title>{title}</title><style>{CSS}</style>"
                f"<h2>{title} ({len(df):,} papers; ADS snapshot {FREEZE}) — click a column to sort</h2>"
                + d.to_html(index=False, escape=False, border=0, na_rep="") + f"<script>{SORT_JS}</script>")

save_html(sample, "sample.html", "JWST galaxy papers using JWST data (the sample)")
save_html(rejected, "rejected.html", "Candidates left out of the sample, with the reason")
print(f"-> general/outputs/sample.csv ({len(sample):,}), rejected.csv ({len(rejected):,}), sample.html, rejected.html")